# 01 — NinaPro DB5 dataset loader

**MATLAB equivalent:** `Dataset_Load.m`

Loads NinaPro DB5 files for subjects S1–S10 and exercises E1–E3, using protocol A1. The subsequent baseline uses **Exercise E1**, retained as `all_E1`.

## Expected dataset layout

Place the dataset under `data/NinaPro_DB5/`:

```text
data/NinaPro_DB5/
├── S1/
│   ├── S1_E1_A1.mat
│   ├── S1_E2_A1.mat
│   └── S1_E3_A1.mat
├── S2/
│   └── ...
└── S10/
```

Download/access the dataset separately; the raw NinaPro data are not included in this repository.

The loader supports standard MATLAB MAT files through SciPy and attempts an HDF5 fallback for MATLAB v7.3 files.

In [1]:
from pathlib import Path
import numpy as np
import joblib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "NinaPro_DB5"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SUBJECTS = 10
N_CHANNELS = 16
FS = 200
WINDOW_SAMPLES = 40
STEP_SAMPLES = 10
CLASS_ORDER = np.arange(13)  # 0 = rest, 1–12 = movements


from scipy.io import loadmat

def load_mat_file(path):
    """Load a MATLAB .mat file, supporting standard MAT and v7.3/HDF5."""
    try:
        data = loadmat(path, squeeze_me=False, struct_as_record=False)
        return {k: v for k, v in data.items() if not k.startswith("__")}
    except (NotImplementedError, ValueError):
        try:
            import h5py
        except ImportError as exc:
            raise ImportError(
                "This file may be MATLAB v7.3. Install h5py: pip install h5py"
            ) from exc

        result = {}
        with h5py.File(path, "r") as f:
            for key in ("emg", "restimulus", "rerepetition"):
                if key in f:
                    result[key] = np.array(f[key])
        return result

def orient_emg(emg):
    """Return EMG as samples × channels."""
    emg = np.asarray(emg)
    if emg.ndim != 2:
        raise ValueError(f"Expected 2-D EMG array, got shape {emg.shape}")
    if emg.shape[1] == N_CHANNELS:
        return emg
    if emg.shape[0] == N_CHANNELS:
        return emg.T
    raise ValueError(
        f"Could not identify the {N_CHANNELS}-channel axis in shape {emg.shape}"
    )

def normalize_loaded_record(raw, file_path):
    required = {"emg", "restimulus", "rerepetition"}
    missing = required.difference(raw)
    if missing:
        raise KeyError(f"{file_path} is missing variables: {sorted(missing)}")

    emg = orient_emg(raw["emg"]).astype(np.float64, copy=False)
    restimulus = np.asarray(raw["restimulus"]).reshape(-1)
    rerepetition = np.asarray(raw["rerepetition"]).reshape(-1)

    if len(emg) != len(restimulus) or len(emg) != len(rerepetition):
        raise ValueError(
            f"Sample-count mismatch in {file_path}: "
            f"EMG={len(emg)}, restimulus={len(restimulus)}, "
            f"rerepetition={len(rerepetition)}"
        )

    return {
        "emg": emg,
        "restimulus": restimulus,
        "rerepetition": rerepetition,
    }

## Load subjects and exercises

The MATLAB loader attempted all three exercises. Here we retain the same intention, but fail clearly if a required file is absent instead of accidentally reusing a previous `loadedData` value.

In [2]:
DB5 = {}
all_E1 = []

missing_files = []

for subject_idx in range(1, N_SUBJECTS + 1):
    subject_name = f"S{subject_idx}"
    DB5[subject_name] = {}

    for exercise_idx in range(1, 4):
        exercise_name = f"E{exercise_idx}"
        filename = f"{subject_name}_{exercise_name}_A1.mat"
        file_path = DATA_ROOT / subject_name / filename

        if not file_path.is_file():
            missing_files.append(str(file_path))
            continue

        raw = load_mat_file(file_path)
        record = normalize_loaded_record(raw, file_path)
        DB5[subject_name][exercise_name] = record
        print(f"Loaded: {filename}")

    if "E1" not in DB5[subject_name]:
        raise FileNotFoundError(
            f"Required E1 file is missing for {subject_name}: "
            f"{DATA_ROOT / subject_name / f'{subject_name}_E1_A1.mat'}"
        )

    all_E1.append(DB5[subject_name]["E1"])

if missing_files:
    print("\\nFiles not loaded (non-E1 exercises may be absent):")
    for item in missing_files:
        print(" -", item)

print(f"\\nDataset loading complete: {len(all_E1)} subjects with E1.")

FileNotFoundError: Required E1 file is missing for S1: /Users/meghmhatre/Downloads/EMG_Hand_Classification_Python/data/NinaPro_DB5/S1/S1_E1_A1.mat

## Save the E1 recordings

The rest of the pipeline uses E1 only. Save that subset as an intermediate artifact.

In [ ]:
joblib.dump(all_E1, OUTPUT_DIR / "all_E1.joblib", compress=3)
print("Saved:", OUTPUT_DIR / "all_E1.joblib")